## Sampling initial conditions
Import libraries

In [ ]:
import os
import sys
import shutil
import numpy as np
import pandas as pd

In [ ]:
sys.path.append(os.path.abspath('./functions'))

from thermos import *
from sampling import *

Define inputs

In [ ]:
# Reading reference solution data
path_to_case = '../'
model = 'thermolib'
scheme = 'strang'
nb_nodes = 1401
cols = ['rho_O','rho_O2','u','Th','Tv']
# Sampling data
bounds_raw = {
    'XO2': [  0.,      1.],
    'p':   [900., 100000.],
    'Th':  [250.,  10000.],
    'Tv':  [250.,  10000.]
}
nb_train_cases = 5120
nb_test_cases = 50
kde_samples_perc = 0.05
extra_perc = 0.1
# Left/right equilibrium conditions
nb_eq_conds = 2
eq_conds_rep = 20

Set paths

In [ ]:
# Path to case
path = os.path.abspath(path_to_case) + f"/pde/{model}/n{nb_nodes}/{scheme}/output/"
# Saving path
saving_path = os.path.abspath("./output/sampling/")
if os.path.exists(saving_path):
    shutil.rmtree(saving_path)
os.makedirs(saving_path, exist_ok=True)

Read solution data

In [ ]:
data = []
for file in os.listdir(path):
    if file.startswith('restart_EU_NLTE') and file.endswith('_0.dat') and file != 'restart_EU_NLTE_0.dat':
        filename = path + '/' + file
        data.append(np.loadtxt(filename))
        # Get left/right equilibrium conditions
        if file == 'restart_EU_NLTE_0_0.dat':
            data_eq = np.take(np.loadtxt(filename), [1,-1], axis=0)
            df_eq = pd.DataFrame(data_eq, columns=cols)
data = np.vstack(data)
# Create dataframe
df = pd.DataFrame(data, columns=cols)
df_eq = pd.DataFrame(data_eq, columns=cols)

Initialize thermodynamics and extract sampling variables: $X = [X_{O_2}, p, T_h, T_v, T_h-T_v]$

In [ ]:
sp_th, mix = init_thermos()

In [ ]:
X = extract_sampling_qoi(df, sp_th, mix)
X_eq = extract_sampling_qoi(df_eq, sp_th, mix)[:,:-1]

Set bounds

In [ ]:
# Compute bounds
phys_bounds = np.array(list(bounds_raw.values())).T
bounds = compute_bounds(
    X,
    phys_bounds=phys_bounds,
    extra_perc=extra_perc
)

In [ ]:
# Force bounds
# Set max dT with (Th_max-Th_min)
bounds[-1,-1] = bounds[1,2] - bounds[0,2]
# Set min Th and Tv
bounds[0,2:4] = bounds_raw['Th'][0]

In [ ]:
# Saving computed bounds
df = pd.DataFrame(bounds, columns=list(bounds_raw.keys())+['dT'])
df.index = ['Min', 'Max']
df.to_csv(saving_path + '/bounds.csv', float_format='%.2f')

In [ ]:
print('Defined bounds:')
df

Construct kde

In [ ]:
X_ = np.take(X, np.arange(X.shape[1]-1), 1)
scaler = construct_scaler(X_)
kde = construct_kde(X_, bandwidth=0.05, scaler=scaler)

Uniform and kde-based sampling

In [ ]:
nb_samples = {
    'train': nb_train_cases - eq_conds_rep * nb_eq_conds,
    'valid': nb_train_cases - eq_conds_rep * nb_eq_conds,
    'test': nb_test_cases - nb_eq_conds
}
bounds_prim = np.take(
    bounds, np.arange(bounds.shape[1]-1), axis=1
)

sampled_data = {
    k: lhs_kde_combined_sampling(
        nb,
        bounds,
        bounds_raw,
        bounds_prim,
        kde,
        scaler,
        kde_samples_perc
    ) for k, nb in nb_samples.items()
}

In [ ]:
# Add samples for left/right equilibrium conditions
sampled_data['test'] = np.vstack([sampled_data['test'], X_eq])
samples_eq = np.tile(X_eq, (eq_conds_rep,1))
for k in ('train', 'valid'):
    sampled_data[k] = np.vstack([sampled_data[k], samples_eq])

In [ ]:
print('Data shapes:')
tf.nest.map_structure(lambda x: x.shape, sampled_data)

Save samples in files

In [ ]:
# Saving
for k, samples in sampled_data.items():
    df = pd.DataFrame(samples, columns=list(bounds_raw.keys()))
    path_samples = saving_path + f'/data/{k}/'
    os.makedirs(path_samples, exist_ok=True)
    df.to_csv(path_samples + '/inp_space.csv', float_format='%.8e', index=False)

Visualize samples

In [ ]:
path_figs = saving_path + '/figs/'
os.makedirs(path_figs, exist_ok=True)

In [ ]:
labels = ["$T_v$ [K]", "$T_h$ [K]"]
Th, Tv = sampled_data['train'][:,2], sampled_data['train'][:,3]

# 3-D plot
plot_hist_3d(
    Tv, Th, labels, figname=path_figs+'/hist_3d.pdf', nb_bins=40
)
# 2-D plot
plot_hist_2d(
    Th, Tv, np.flip(labels), figname=path_figs+'/hist_2d.pdf', nb_bins=40
)
# 1-D plot
for i, k in enumerate(bounds_raw.keys()):
    plot_hist_1d(
        sampled_data['train'][:,i], k, figname=path_figs+f'/hist_1d_{k}_kde.pdf', nb_bins=100
    )
    plot_hist_1d(
        X[:,i], k, figname=path_figs+f'/hist_1d_{k}_data.pdf', nb_bins=100
    )